# Experiment summary: choosing the Best model

This notebook compares the three Runs (one per candidate), shows their TensorBoard curves (Optional 2), picks the **Best model** and holds the write-up (TODO 2).
Run it after all three `1_train_model_<candidate>.ipynb` notebooks have finished. Each of them writes `results/<candidate>.json`.

| Candidate | Pretrained model | Notebook |
|---|---|---|
| `efficientdet_d1` | EfficientDet D1 640x640 (baseline) | [1_train_model_efficientdet_d1.ipynb](1_train_model_efficientdet_d1.ipynb) |
| `ssd_mobilenet_v2_fpnlite_640` | SSD MobileNet V2 FPNLite 640x640 | [1_train_model_ssd_mobilenet_v2_fpnlite_640.ipynb](1_train_model_ssd_mobilenet_v2_fpnlite_640.ipynb) |
| `ssd_resnet50_v1_fpn_640` | SSD ResNet50 V1 FPN 640x640 (RetinaNet50) | [1_train_model_ssd_resnet50_v1_fpn_640.ipynb](1_train_model_ssd_resnet50_v1_fpn_640.ipynb) |

**Shared recipe (identical for every Run):** 2,000 steps at batch 8, momentum 0.9, cosine decay from a base learning rate of 0.02 with 200 warm-up steps from 0.002, fp32,
random horizontal flip + random scale/crop/pad to 640, fine-tuned from COCO detection checkpoints, `num_classes: 4` (label map ids 1, 2, 4).
Only the architecture changes between Runs. See `docs/adr/0001` and `docs/adr/0002` for why.

In [ ]:
%pip install -q tensorboard pandas matplotlib

In [ ]:
import json
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import pandas as pd

import tb_helpers as tb

# On-demand price of ml.g5.xlarge training in us-east-1 when this was written; check the AWS pricing page.
G5_XLARGE_TRAINING_USD_PER_HOUR = 1.408

registry = json.loads(Path("candidates.json").read_text())["candidates"]
runs = {}
for entry in registry:
    path = Path("results") / f"{entry['key']}.json"
    if path.is_file():
        runs[entry["key"]] = json.loads(path.read_text())
    else:
        print(f"Missing {path}: run 1_train_model_{entry['key']}.ipynb first")
print("Runs found:", list(runs))

In [ ]:
# Download each Run's TensorBoard events from S3 and load them as tidy frames.
s3 = boto3.client("s3")
scalars = {}
for key, run in runs.items():
    bucket, _, prefix = run["tensorboard_s3_prefix"].removeprefix("s3://").partition("/")
    dirs = tb.download_event_files(s3, bucket, prefix, Path("/tmp/tb") / key)
    scalars[key] = {split: tb.load_scalars(d) for split, d in dirs.items()}
    print(key, {split: len(df) for split, df in scalars[key].items()})

## Comparison table

mAP is COCO mAP@[.5:.95] on the validation split. The training loss is the mean over the last 10% of steps, and the validation loss comes from the single evaluation of the final checkpoint.

In [ ]:
empty = pd.DataFrame(columns=["step", "tag", "value"])
summaries = {
    key: tb.summarize_run(s.get("train", empty), s.get("eval", empty)) for key, s in scalars.items()
}
table = pd.DataFrame(summaries).T
table["train minutes"] = [runs[k]["training_seconds"] / 60 for k in table.index]
table["approx cost USD"] = [runs[k]["billable_seconds"] / 3600 * G5_XLARGE_TRAINING_USD_PER_HOUR for k in table.index]
table = table.sort_values("mAP", ascending=False)

columns = ["mAP", "mAP@0.5", "mAP small", "mAP medium", "mAP large", "AR@100",
           "train Loss/total_loss", "val Loss/total_loss",
           "train Loss/classification_loss", "val Loss/classification_loss",
           "train Loss/localization_loss", "val Loss/localization_loss",
           "train minutes", "approx cost USD"]
table[columns].style.format("{:.3f}", na_rep="n/a")

In [ ]:
best_key = tb.pick_best(summaries)
best = {"candidate": best_key, "model_artifact": runs[best_key]["model_artifact"], "mAP": summaries[best_key]["mAP"]}
Path("results").mkdir(exist_ok=True)
Path("results/best_model.json").write_text(json.dumps(best, indent=2) + "\n")
table[columns].to_csv("results/summary.csv")
print(json.dumps(best, indent=2))

## TensorBoard graphs (Optional 2)

In [ ]:
def plot_train_curves(tags, title):
    fig, axes = plt.subplots(1, len(tags), figsize=(5 * len(tags), 3.5), squeeze=False)
    for ax, tag in zip(axes[0], tags):
        for key, s in scalars.items():
            df = s.get("train", empty)
            df = df[df["tag"] == tag]
            ax.plot(df["step"], df["value"], label=key)
        ax.set_title(tag)
        ax.set_xlabel("step")
        ax.grid(alpha=0.3)
    axes[0][0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    return fig

plot_train_curves(["Loss/total_loss", "Loss/classification_loss", "Loss/localization_loss"], "Training losses")
plot_train_curves(["learning_rate"], "Learning-rate schedule (shared by every Run)");

In [ ]:
# Training vs validation loss at the end of each Run
loss_parts = ["classification_loss", "localization_loss", "total_loss"]
fig, axes = plt.subplots(1, len(loss_parts), figsize=(15, 3.5))
for ax, part in zip(axes, loss_parts):
    table[[f"train Loss/{part}", f"val Loss/{part}"]].plot.bar(ax=ax, rot=15)
    ax.set_title(part)
    ax.legend(["train", "validation"], fontsize=8)
fig.tight_layout()

ax = table[["mAP", "mAP@0.5", "mAP small", "mAP medium", "mAP large"]].plot.bar(figsize=(10, 3.5), rot=15)
ax.set_title("COCO metrics on the validation split");

## Write-up (TODO 2)

> Fill in the bracketed parts once the three Runs have finished; the numbers come from the table above.

### Experiments

We fine-tuned three COCO-pretrained detectors from the TF2 Model Zoo with one shared training recipe, so the architecture is the only thing that changes between Runs:
EfficientDet D1 (the project's default, used as the baseline), SSD MobileNet V2 FPNLite 640 and SSD ResNet50 V1 FPN 640 (RetinaNet50).

The starter configuration had two problems, which we fixed for every Run:

1. **Cyclists were not learnable.** The label map uses ids 1, 2 and **4**, but the config declared `num_classes: 3`. The OD API one-hot encodes class ids with depth `num_classes`, so cyclist boxes would be treated as background. We set `num_classes: 4` and left id 3 unused.
2. **The learning rate never left warm-up.** The EfficientDet recipe (2,500 warm-up steps, 300k cosine steps, base 0.08 at batch 128 on TPU) was used for a 2,000-step job at batch 8. We sized the schedule to the real job: 200 warm-up steps, cosine decay to step 2,000, base 0.02.

### Results

| Model | mAP | mAP@0.5 | mAP small | train loss | val loss | train time |
|---|---|---|---|---|---|---|
| EfficientDet D1 | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |
| SSD MobileNet V2 FPNLite | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |
| SSD ResNet50 V1 FPN | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |

**Best model: [ ]**, chosen by highest mAP. Within 0.01 mAP of the leader, the better small-object mAP decides, because most pedestrians and cyclists are small.

### How does the validation loss compare to the training loss?

[Compare the numbers. We expect validation loss to be higher than training loss for every model.]

### Did you expect such behavior from the losses / metrics?

[Expected: yes. (1) The validation scenes are different drives, under different weather and lighting, from the training scenes, so some generalization gap is normal.
(2) Augmentation is applied only during training, which pushes training loss *up*, so a large gap means real overfitting rather than an augmentation effect.
(3) With only 2,000 steps at batch 8 (about 16k images seen), every model is still under-trained; absolute mAP will be well below COCO-level numbers, and small objects (mAP small) will lag far behind large ones.]

### What can you do to further improve the performance of the tested models?

[Ideas to rank once the numbers are in: train longer (the loss curves are still falling at step 2,000); add lighting augmentation (`random_adjust_brightness`, `random_adjust_contrast`) for night scenes;
a higher input resolution or a two-stage detector (Faster R-CNN) for small, distant pedestrians and cyclists; deal with class imbalance (cyclists are rare) through focal-loss tuning or oversampling;
evaluate intermediate checkpoints to pick the best step instead of the last one.]